<!--nav--> [🗺 Learning path](README.md) · **9/20** · ◀ [Colab Pro A100 Max Training](./Colab_Pro_A100_Max_Training.ipynb) · [DPO Training](./DPO_Training.ipynb) ▶

# Post-Training: See Every Stage Transform a Model

This notebook has one goal: **watch a raw language model become a helpful assistant.**

We run every stage, generate text at each checkpoint, and see exactly what changed.

```
Stage 0: BASE MODEL        → completes text like autocomplete, can't follow instructions
         "What is gravity?" → "What is gravity? What is the meaning of gravity? In..."
              |
Stage 1: SFT (fine-tune)   → learns instruction → response format
         "What is gravity?" → "Gravity is a fundamental force that attracts..."
              |
Stage 2: DPO (align)       → learns WHICH answers humans prefer
         "What is gravity?" → "Gravity is the force of attraction between objects
                               with mass. It keeps planets in orbit and causes
                               objects to fall to the ground..."
```

**Runtime:** T4 GPU (free on Kaggle or Colab) — takes ~10-15 minutes total

## Step 1: Install & Setup

In [ ]:
!pip install -q transformers trl datasets peft accelerate

In [ ]:
import torch, os, time, gc
os.environ["WANDB_DISABLED"] = "true"

assert torch.cuda.is_available(), "Need a GPU!"
print("GPU:", torch.cuda.get_device_name(0))
print("Memory: %.0f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

MODEL = "Qwen/Qwen2.5-0.5B"  # small enough to run fast, big enough to show real behavior
print("Model:", MODEL)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Test prompts we'll use at every stage to see the transformation
TEST_PROMPTS = [
    "What is gravity?",
    "How do I make scrambled eggs?",
    "Explain Python lists to a beginner.",
    "Why is the sky blue?",
    "Is it true that we only use 10% of our brain?",
]

def generate(model, prompt, max_tokens=150):
    """Generate a response from a model."""
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens, temperature=0.7,
                             do_sample=True, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def generate_raw(model, prompt, max_tokens=100):
    """Generate without chat template — shows raw completion behavior."""
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_tokens, temperature=0.7,
                             do_sample=True, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# Store all responses for final comparison
all_responses = {}

print("Ready. We'll test with %d prompts at each stage." % len(TEST_PROMPTS))

---
## Stage 0: The Base Model (Before Post-Training)

A pre-trained model is like a student who has **read millions of books**
but has **never had a conversation**.

It knows facts, grammar, and patterns — but if you ask it a question,
it just continues the text like autocomplete.

```
You type:    "What is gravity?"
It thinks:   "This looks like the start of a paragraph or essay..."
It outputs:  "What is gravity? What is the force of gravity? How does
              gravity work? These are questions that have puzzled..."

It doesn't ANSWER. It CONTINUES.
```

In [ ]:
# Load the base model
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.bfloat16, trust_remote_code=True
).to("cuda")
base_model.eval()

print("=" * 70)
print("  STAGE 0: BASE MODEL (pre-trained, no post-training)")
print("  The model just CONTINUES text. It doesn't know how to ANSWER.")
print("=" * 70)

all_responses["base"] = {}

for prompt in TEST_PROMPTS:
    # Show RAW completion (no chat template)
    raw_response = generate_raw(base_model, prompt, max_tokens=80)
    # Also try with chat template
    chat_response = generate(base_model, prompt, max_tokens=80)
    
    all_responses["base"][prompt] = chat_response
    
    print("\nQ: %s" % prompt)
    print("  [raw completion]:  %s" % raw_response[:200])
    print("  [chat template]:   %s" % chat_response[:200])
    print("-" * 70)

print("\nNotice: The base model often repeats the question, rambles,")
print("or generates unrelated text. It hasn't learned to be an assistant yet.")

del base_model
gc.collect()
torch.cuda.empty_cache()

---
## Stage 1: SFT (Supervised Fine-Tuning)

### What SFT does

We show the model thousands of examples of:
```
User: <question>
Assistant: <good answer>
```

The model learns: when someone asks a question, **generate an answer** (don't just continue).

### How it learns

SFT uses the same loss as pre-training: **next-token prediction**.
But instead of predicting the next word in a book, it predicts
the next word in an instruction-response pair.

```
Training example:
  Input:  "User: What is gravity? Assistant:"
  Target: "Gravity is a fundamental force..."
  Loss:   cross-entropy on each token of the answer

After thousands of examples, the model learns:
  1. The format: User asks → Assistant answers
  2. The style: be helpful, clear, concise
  3. The content: use its knowledge to answer
```

### What SFT does NOT do

SFT teaches the model to give **an** answer.
It does NOT teach it to give the **best** answer.
That's what Stage 2 (DPO) is for.

In [ ]:
from datasets import load_dataset

# Load instruction-tuning data
sft_data = load_dataset("tatsu-lab/alpaca", split="train")
sft_data = sft_data.shuffle(seed=42).select(range(2000))

def format_sft(example):
    prompt = example["instruction"]
    if example.get("input"):
        prompt += "\n" + example["input"]
    messages = [
        {"role": "user", "content": prompt},
        {"role": "assistant", "content": example["output"]},
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

sft_formatted = sft_data.map(format_sft, remove_columns=sft_data.column_names)

print("SFT dataset: %d instruction-response pairs" % len(sft_formatted))
print("\nExample:")
print(sft_formatted[0]["text"][:300])

In [ ]:
from trl import SFTConfig, SFTTrainer
from peft import LoraConfig

# Load fresh model for SFT
sft_model = AutoModelForCausalLM.from_pretrained(
    MODEL, torch_dtype=torch.bfloat16, trust_remote_code=True
).to("cuda")

lora_config = LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    bias="none", task_type="CAUSAL_LM",
)

sft_trainer = SFTTrainer(
    model=sft_model,
    args=SFTConfig(
        output_dir="./sft_output",
        num_train_epochs=2,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        learning_rate=2e-4,
        warmup_steps=20,
        logging_steps=10,
        bf16=True,
        gradient_checkpointing=True,
        max_length=512,
        dataset_text_field="text",
        report_to="none",
        save_strategy="no",
    ),
    train_dataset=sft_formatted,
    processing_class=tokenizer,
    peft_config=lora_config,
)

print("Starting SFT...")
sft_start = time.time()
sft_trainer.train()
sft_time = time.time() - sft_start
print("SFT done in %.0f seconds!" % sft_time)

# Save SFT checkpoint (we'll load it for DPO)
sft_trainer.save_model("./sft_output/final")
tokenizer.save_pretrained("./sft_output/final")

In [ ]:
# Test SFT model
sft_model.eval()

print("=" * 70)
print("  STAGE 1: AFTER SFT")
print("  The model now ANSWERS questions instead of continuing text.")
print("=" * 70)

all_responses["sft"] = {}

for prompt in TEST_PROMPTS:
    response = generate(sft_model, prompt)
    all_responses["sft"][prompt] = response
    print("\nQ: %s" % prompt)
    print("A: %s" % response[:300])
    print("-" * 70)

print("\nNotice: The model now gives real answers! But they might be")
print("short, generic, or not the BEST possible answer. That's next.")

del sft_trainer
gc.collect()
torch.cuda.empty_cache()

---
## Stage 2: DPO (Direct Preference Optimization)

### What DPO does

SFT taught the model to answer. DPO teaches it which answers are **better**.

```
Prompt: "What is gravity?"

Chosen (good):    "Gravity is the force of attraction between objects
                   with mass. It's what keeps planets in orbit and
                   causes objects to fall to the ground."              <- detailed, accurate

Rejected (bad):   "Gravity is a thing. It makes stuff fall."          <- lazy, vague

DPO loss: make P(chosen) > P(rejected)
```

### How DPO works (the core idea)

```
For each preference pair (prompt, chosen, rejected):

  1. Compute log-probability of chosen response under our model
  2. Compute log-probability of rejected response under our model
  3. Compute same log-probs under the reference model (frozen SFT model)
  4. Push our model to increase the gap:
     
     loss = -log(sigmoid(β * [(log P_model(chosen) - log P_ref(chosen))
                             - (log P_model(rejected) - log P_ref(rejected))]))
     
  The β parameter (0.1) controls how aggressively the model changes.
  The reference model prevents the model from changing too much.
```

### Why DPO over RLHF?

```
RLHF (old):  Train reward model → Run PPO → unstable, complex
DPO  (new):  Train directly on preference pairs → stable, simple

Same result, 1/3 the code, way easier to train.
```

In [ ]:
from datasets import Dataset

# Preference pairs: chosen (good) vs rejected (bad) responses
# These teach the model QUALITY differences
preference_pairs = [
    {
        "prompt": "What is gravity?",
        "chosen": "Gravity is the fundamental force of attraction between objects with mass. It's described by Newton's law of universal gravitation and Einstein's general relativity. In everyday life, gravity keeps us on the ground, makes objects fall, and keeps planets orbiting stars.",
        "rejected": "Gravity is a force. It makes things fall down."
    },
    {
        "prompt": "How do I make scrambled eggs?",
        "chosen": "Crack 2-3 eggs into a bowl and whisk with a pinch of salt and pepper. Heat a non-stick pan over medium-low heat with a tablespoon of butter. Pour in the eggs and gently stir with a spatula, pushing curds from the edges toward the center. Remove from heat when slightly underdone — they'll continue cooking. Serve immediately.",
        "rejected": "Put eggs in a pan and cook them."
    },
    {
        "prompt": "Explain Python lists to a beginner.",
        "chosen": "A Python list is an ordered collection that can hold multiple items. Create one with square brackets: `colors = ['red', 'blue', 'green']`. Access items by position: `colors[0]` gives 'red'. Add items with `colors.append('yellow')`. Lists can hold any type — numbers, strings, even other lists. They're one of the most useful tools in Python.",
        "rejected": "Lists are data structures in Python. They store elements."
    },
    {
        "prompt": "Why is the sky blue?",
        "chosen": "The sky appears blue because of Rayleigh scattering. When sunlight enters Earth's atmosphere, it collides with gas molecules. Blue light has a shorter wavelength, so it scatters more than other colors in all directions. When you look up, you see this scattered blue light coming from everywhere in the sky.",
        "rejected": "The sky is blue because of the atmosphere."
    },
    {
        "prompt": "Is it true that we only use 10% of our brain?",
        "chosen": "No, this is a myth. Brain imaging studies show that we use virtually all parts of our brain, and most of the brain is active most of the time. Different areas handle different functions — motor control, vision, language, memory. Even during sleep, areas like the frontal cortex and somatosensory areas are active.",
        "rejected": "Yes, we only use 10% of our brain. The rest is unused potential."
    },
    {
        "prompt": "What is machine learning?",
        "chosen": "Machine learning is a branch of AI where computers learn patterns from data instead of being explicitly programmed. For example, a spam filter learns from thousands of labeled emails what spam looks like. The three main types are supervised learning (labeled data), unsupervised learning (finding patterns), and reinforcement learning (learning from rewards).",
        "rejected": "Machine learning is AI stuff. Computers learn things."
    },
    {
        "prompt": "How does the internet work?",
        "chosen": "When you visit a website, your browser sends a request through your ISP to a DNS server, which translates the domain name to an IP address. The request travels through routers to the web server, which sends back the page data in packets. These packets may take different routes but are reassembled by your browser into the page you see. This all happens using the TCP/IP protocol stack.",
        "rejected": "The internet uses wifi and cables to connect computers."
    },
    {
        "prompt": "What is photosynthesis?",
        "chosen": "Photosynthesis is how plants convert light energy into chemical energy (food). In the chloroplasts of plant cells, chlorophyll absorbs sunlight and uses it to convert carbon dioxide from the air and water from the soil into glucose and oxygen. The equation: 6CO2 + 6H2O + light energy → C6H12O6 + 6O2.",
        "rejected": "Plants use sunlight to make food somehow."
    },
    {
        "prompt": "Explain recursion in programming.",
        "chosen": "Recursion is when a function calls itself to solve a problem by breaking it into smaller sub-problems. Each call handles a smaller piece until reaching a base case that stops the recursion. Example: factorial(5) = 5 × factorial(4) = 5 × 4 × factorial(3) = ... = 5 × 4 × 3 × 2 × 1 = 120. Every recursive function needs a base case (factorial(1) = 1) to avoid infinite loops.",
        "rejected": "Recursion is when a function calls itself. It's complicated."
    },
    {
        "prompt": "Why do we dream?",
        "chosen": "Scientists don't fully agree, but leading theories include: memory consolidation (the brain processes and stores important information from the day), emotional regulation (dreams help process difficult emotions), and threat simulation (practicing responses to dangers). During REM sleep, the brain is highly active while the body is paralyzed, creating vivid dream experiences.",
        "rejected": "Nobody really knows why we dream."
    },
]

def format_dpo(pair):
    user_msg = {"role": "user", "content": pair["prompt"]}
    return {
        "prompt": [user_msg],
        "chosen": [user_msg, {"role": "assistant", "content": pair["chosen"]}],
        "rejected": [user_msg, {"role": "assistant", "content": pair["rejected"]}],
    }

dpo_dataset = Dataset.from_list([format_dpo(p) for p in preference_pairs])

print("DPO dataset: %d preference pairs" % len(dpo_dataset))
print("\nExample pair:")
print("  Prompt:   %s" % preference_pairs[0]["prompt"])
print("  Chosen:   %s" % preference_pairs[0]["chosen"][:80])
print("  Rejected: %s" % preference_pairs[0]["rejected"][:80])

In [ ]:
from trl import DPOConfig, DPOTrainer

# DPO trains on top of the SFT model
dpo_trainer = DPOTrainer(
    model=sft_model,  # start from SFT checkpoint
    args=DPOConfig(
        output_dir="./dpo_output",
        beta=0.1,                        # controls how much to trust preferences
        num_train_epochs=3,              # 3 epochs over 10 pairs
        per_device_train_batch_size=2,
        gradient_accumulation_steps=2,
        learning_rate=5e-5,              # lower than SFT — gentle alignment
        warmup_steps=5,
        logging_steps=1,                 # log every step to see DPO dynamics
        bf16=True,
        gradient_checkpointing=True,
        report_to="none",
        save_strategy="no",
        max_length=512,
        remove_unused_columns=False,
    ),
    train_dataset=dpo_dataset,
    processing_class=tokenizer,
)

print("Starting DPO...")
print("  beta=0.1 — moderate preference strength")
print("  lr=5e-5 — gentle updates (5x lower than SFT)")

dpo_start = time.time()
dpo_trainer.train()
dpo_time = time.time() - dpo_start
print("DPO done in %.0f seconds!" % dpo_time)

# Save DPO model
dpo_trainer.save_model("./dpo_output/final")
tokenizer.save_pretrained("./dpo_output/final")

In [ ]:
# Test DPO model
dpo_model = dpo_trainer.model
dpo_model.eval()

print("=" * 70)
print("  STAGE 2: AFTER DPO")
print("  The model now gives BETTER, more detailed answers.")
print("=" * 70)

all_responses["dpo"] = {}

for prompt in TEST_PROMPTS:
    response = generate(dpo_model, prompt)
    all_responses["dpo"][prompt] = response
    print("\nQ: %s" % prompt)
    print("A: %s" % response[:300])
    print("-" * 70)

del dpo_trainer
gc.collect()
torch.cuda.empty_cache()

---
## Evaluation: How Good Are These Responses?

We use **three methods** to measure quality — the same methods used in production:

```
Method 1: Rule-Based Scoring        → automated metrics (word count, specificity, structure)
Method 2: LLM-as-Judge              → the model itself rates responses on 5 dimensions
Method 3: Pairwise LLM Judge        → the model picks a winner: Base vs SFT vs DPO
```

In [ ]:
import re, numpy as np
from IPython.display import HTML, display

# ============================================================
# METHOD 1: Rule-Based Scoring (5 dimensions, 0-10 each)
# ============================================================

def score_response(response, prompt):
    """Score a response on 5 quality dimensions."""
    words = response.split()
    sentences = [s.strip() for s in re.split(r'[.!?]+', response) if s.strip()]
    unique_words = set(w.lower() for w in words)
    
    # 1. Completeness (longer = more complete, with diminishing returns)
    wc = len(words)
    completeness = min(10, wc / 8)
    
    # 2. Specificity (numbers, examples, technical terms, causal words)
    has_numbers = len(re.findall(r'\d+', response))
    has_examples = len(re.findall(r'(?:for example|such as|like|e\.g\.|including)', response, re.I))
    long_words = sum(1 for w in words if len(w) > 7)
    specificity = min(10, (has_numbers * 1.5 + has_examples * 2 + long_words * 0.3))
    
    # 3. Structure (sentences, transitions, enumeration)
    transitions = len(re.findall(r'(?:however|because|therefore|first|second|also|additionally|moreover)', response, re.I))
    has_steps = len(re.findall(r'(?:step \d|^\d\.|\d\))', response, re.I | re.M))
    structure = min(10, len(sentences) * 0.8 + transitions * 1.5 + has_steps * 2)
    
    # 4. Relevance (overlap between prompt words and response)
    prompt_words = set(w.lower() for w in prompt.split() if len(w) > 3)
    overlap = len(prompt_words & unique_words)
    relevance = min(10, overlap * 2 + (2 if wc > 15 else 0))
    
    # 5. Helpfulness (actionable language, directness)
    actionable = len(re.findall(r'(?:you can|try|use|make sure|remember|important)', response, re.I))
    direct = 1 if not response.startswith(("Well", "Hmm", "So,", "I think")) else 0
    helpfulness = min(10, actionable * 2 + direct * 3 + (3 if wc > 30 else 0))
    
    return {
        "Completeness": round(completeness, 1),
        "Specificity": round(specificity, 1),
        "Structure": round(structure, 1),
        "Relevance": round(relevance, 1),
        "Helpfulness": round(helpfulness, 1),
    }

# Score all responses
scores = {}
for stage in ["base", "sft", "dpo"]:
    scores[stage] = {}
    for prompt in TEST_PROMPTS:
        scores[stage][prompt] = score_response(all_responses[stage][prompt], prompt)

# Compute averages per dimension per stage
avg_scores = {}
dimensions = ["Completeness", "Specificity", "Structure", "Relevance", "Helpfulness"]
for stage in ["base", "sft", "dpo"]:
    avg_scores[stage] = {}
    for dim in dimensions:
        avg_scores[stage][dim] = np.mean([scores[stage][p][dim] for p in TEST_PROMPTS])

print("Rule-Based Scores (0-10):\n")
print("%-15s" % "Dimension", end="")
for label in ["Base", "SFT", "DPO"]:
    print("%-10s" % label, end="")
print()
print("-" * 45)
for dim in dimensions:
    print("%-15s" % dim, end="")
    for stage in ["base", "sft", "dpo"]:
        print("%-10.1f" % avg_scores[stage][dim], end="")
    print()

In [ ]:
# ============================================================
# METHOD 2: LLM-as-Judge — The DPO model judges ALL responses
# ============================================================

print("=" * 70)
print("  LLM-AS-JUDGE: Using the DPO model to score all responses")
print("=" * 70)

JUDGE_PROMPT = """Rate this response to the question on a scale of 1-10 for each dimension.
Question: {question}
Response: {response}

Score each dimension (just the number):
Accuracy: 
Clarity: 
Detail: 
Helpfulness: 
Overall: """

judge_model = dpo_model  # reuse the DPO model as judge

def llm_judge(question, response):
    """Use the model itself as a judge to score a response."""
    prompt_text = JUDGE_PROMPT.format(question=question, response=response[:300])
    messages = [{"role": "user", "content": prompt_text}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(judge_model.device)
    with torch.no_grad():
        out = judge_model.generate(**inputs, max_new_tokens=50, temperature=0.1,
                                    do_sample=True, pad_token_id=tokenizer.pad_token_id)
    result = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    
    # Extract numbers from judge output
    numbers = re.findall(r'(\d+(?:\.\d+)?)', result)
    dims = ["Accuracy", "Clarity", "Detail", "Helpfulness", "Overall"]
    judge_scores = {}
    for i, dim in enumerate(dims):
        if i < len(numbers):
            judge_scores[dim] = min(10, max(1, float(numbers[i])))
        else:
            judge_scores[dim] = 5.0  # default if parsing fails
    return judge_scores

# Run LLM judge on all responses
llm_scores = {}
for stage in ["base", "sft", "dpo"]:
    llm_scores[stage] = {}
    for prompt in TEST_PROMPTS:
        llm_scores[stage][prompt] = llm_judge(prompt, all_responses[stage][prompt])
    print("  Judged %s stage" % stage)

# Average LLM judge scores
avg_llm = {}
judge_dims = ["Accuracy", "Clarity", "Detail", "Helpfulness", "Overall"]
for stage in ["base", "sft", "dpo"]:
    avg_llm[stage] = {}
    for dim in judge_dims:
        avg_llm[stage][dim] = np.mean([llm_scores[stage][p][dim] for p in TEST_PROMPTS])

print("\nLLM Judge Scores (1-10):\n")
print("%-15s" % "Dimension", end="")
for label in ["Base", "SFT", "DPO"]:
    print("%-10s" % label, end="")
print()
print("-" * 45)
for dim in judge_dims:
    print("%-15s" % dim, end="")
    for stage in ["base", "sft", "dpo"]:
        print("%-10.1f" % avg_llm[stage][dim], end="")
    print()

In [ ]:
# ============================================================
# METHOD 3: Pairwise LLM Judge — pick a winner
# ============================================================

import random

PAIRWISE_PROMPT = """Compare these two responses and pick the better one.
Question: {question}

Response A: {response_a}

Response B: {response_b}

Which is better? Consider accuracy, detail, and helpfulness.
Answer with ONLY the letter A or B."""

def pairwise_judge(question, resp_a, resp_b):
    """LLM picks a winner between two responses."""
    # Randomize order to avoid position bias
    swap = random.random() > 0.5
    if swap:
        resp_a, resp_b = resp_b, resp_a
    
    prompt_text = PAIRWISE_PROMPT.format(
        question=question, response_a=resp_a[:250], response_b=resp_b[:250]
    )
    messages = [{"role": "user", "content": prompt_text}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512).to(judge_model.device)
    with torch.no_grad():
        out = judge_model.generate(**inputs, max_new_tokens=5, temperature=0.1,
                                    do_sample=True, pad_token_id=tokenizer.pad_token_id)
    result = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    
    pick = "A" if "A" in result[:3] else "B"
    if swap:
        pick = "B" if pick == "A" else "A"
    return pick

# Run pairwise comparisons
matchups = [("base", "sft"), ("sft", "dpo"), ("base", "dpo")]
wins = {"base": 0, "sft": 0, "dpo": 0}

print("Pairwise LLM Judge Results:\n")
for stage_a, stage_b in matchups:
    a_wins, b_wins = 0, 0
    for prompt in TEST_PROMPTS:
        pick = pairwise_judge(prompt, all_responses[stage_a][prompt], all_responses[stage_b][prompt])
        if pick == "A":
            a_wins += 1
            wins[stage_a] += 1
        else:
            b_wins += 1
            wins[stage_b] += 1
    print("  %s vs %s: %s wins %d, %s wins %d" % (
        stage_a.upper(), stage_b.upper(), stage_a.upper(), a_wins, stage_b.upper(), b_wins))

print("\nTotal wins: Base=%d, SFT=%d, DPO=%d" % (wins["base"], wins["sft"], wins["dpo"]))

---
## Interactive Visualizations

In [ ]:
# ============================================================
# VISUALIZATION 1: Interactive Radar Chart (SVG + JS)
# ============================================================

import math, json

def make_radar_svg(rule_scores, llm_scores_avg):
    """Build an interactive SVG radar chart comparing Base/SFT/DPO."""
    # Use rule-based scores for the radar
    dims = ["Completeness", "Specificity", "Structure", "Relevance", "Helpfulness"]
    stages = [("base", "Base", "#f87171"), ("sft", "SFT", "#fbbf24"), ("dpo", "DPO", "#3fb950")]
    
    cx, cy, r = 200, 200, 150
    n = len(dims)
    angles = [i * 2 * math.pi / n - math.pi / 2 for i in range(n)]
    
    # Grid circles
    grid_svg = ""
    for level in [2, 4, 6, 8, 10]:
        cr = r * level / 10
        grid_svg += '<circle cx="%d" cy="%d" r="%.0f" fill="none" stroke="#30363d" stroke-width="1" opacity="0.5"/>' % (cx, cy, cr)
        if level in [5, 10]:
            grid_svg += '<text x="%d" y="%.0f" fill="#8b949e" font-size="10" text-anchor="middle">%d</text>' % (cx, cy - cr - 3, level)
    
    # Axis lines + labels
    for i, (angle, dim) in enumerate(zip(angles, dims)):
        x2 = cx + r * math.cos(angle)
        y2 = cy + r * math.sin(angle)
        grid_svg += '<line x1="%d" y1="%d" x2="%.0f" y2="%.0f" stroke="#30363d" stroke-width="1"/>' % (cx, cy, x2, y2)
        lx = cx + (r + 25) * math.cos(angle)
        ly = cy + (r + 25) * math.sin(angle)
        anchor = "middle" if abs(math.cos(angle)) < 0.3 else ("start" if math.cos(angle) > 0 else "end")
        grid_svg += '<text x="%.0f" y="%.0f" fill="#c9d1d9" font-size="11" font-weight="600" text-anchor="%s">%s</text>' % (lx, ly + 4, anchor, dim)
    
    # Data polygons
    polys = ""
    for stage_key, label, color in stages:
        points = []
        for i, angle in enumerate(angles):
            val = rule_scores[stage_key][dims[i]]
            px = cx + r * (val / 10) * math.cos(angle)
            py = cy + r * (val / 10) * math.sin(angle)
            points.append("%.1f,%.1f" % (px, py))
        pts_str = " ".join(points)
        polys += '<polygon class="radar-%s" points="%s" fill="%s" fill-opacity="0.15" stroke="%s" stroke-width="2.5"/>' % (stage_key, pts_str, color, color)
        # Dots
        for pt in points:
            x, y = pt.split(",")
            polys += '<circle class="radar-%s" cx="%s" cy="%s" r="4" fill="%s"/>' % (stage_key, x, y, color)
    
    svg = '''
    <div style="background:#0d1117;border:1px solid #30363d;border-radius:16px;padding:24px;max-width:500px;margin:20px auto;">
      <div style="text-align:center;margin-bottom:12px;">
        <span style="color:#c9d1d9;font-size:15px;font-weight:700;">Rule-Based Quality Scores</span>
      </div>
      <div style="display:flex;justify-content:center;gap:16px;margin-bottom:12px;">
        <button onclick="toggleRadar('base')" style="background:#f8717133;border:2px solid #f87171;color:#f87171;padding:4px 14px;border-radius:8px;cursor:pointer;font-weight:600;font-size:12px;">Base</button>
        <button onclick="toggleRadar('sft')" style="background:#fbbf2433;border:2px solid #fbbf24;color:#fbbf24;padding:4px 14px;border-radius:8px;cursor:pointer;font-weight:600;font-size:12px;">SFT</button>
        <button onclick="toggleRadar('dpo')" style="background:#3fb95033;border:2px solid #3fb950;color:#3fb950;padding:4px 14px;border-radius:8px;cursor:pointer;font-weight:600;font-size:12px;">DPO</button>
        <button onclick="toggleRadar('all')" style="background:#818cf833;border:2px solid #818cf8;color:#818cf8;padding:4px 14px;border-radius:8px;cursor:pointer;font-weight:600;font-size:12px;">All</button>
      </div>
      <svg viewBox="0 0 400 400" style="width:100%%;max-width:400px;display:block;margin:0 auto;">
        %s
        %s
      </svg>
    </div>
    <script>
    function toggleRadar(stage) {
      ['base','sft','dpo'].forEach(s => {
        document.querySelectorAll('.radar-'+s).forEach(el => {
          el.style.opacity = (stage==='all' || stage===s) ? '1' : '0.08';
          el.style.transition = 'opacity 0.4s ease';
        });
      });
    }
    </script>
    ''' % (grid_svg, polys)
    return svg

display(HTML(make_radar_svg(avg_scores, avg_llm)))

In [ ]:
# ============================================================
# VISUALIZATION 2: Interactive Response Comparison Cards
# ============================================================

cards = ""
for i, prompt in enumerate(TEST_PROMPTS):
    base_r = all_responses["base"][prompt][:250]
    sft_r = all_responses["sft"][prompt][:250]
    dpo_r = all_responses["dpo"][prompt][:250]
    
    # Rule-based total score per stage
    base_total = sum(scores["base"][prompt].values())
    sft_total = sum(scores["sft"][prompt].values())
    dpo_total = sum(scores["dpo"][prompt].values())
    
    cards += '''
    <div class="qcard" data-idx="%d" style="background:#161b22;border:1px solid #30363d;border-radius:12px;padding:20px;margin-bottom:16px;%s">
      <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:12px;cursor:pointer;" onclick="this.parentElement.querySelector('.qbody').style.display=this.parentElement.querySelector('.qbody').style.display==='none'?'block':'none'">
        <span style="color:#a78bfa;font-weight:700;font-size:14px;">Q: %s</span>
        <span style="color:#8b949e;font-size:12px;">click to expand/collapse</span>
      </div>
      <div class="qbody">
        <div style="display:grid;grid-template-columns:1fr 1fr 1fr;gap:12px;">
          <div style="border-left:3px solid #f87171;padding-left:12px;">
            <div style="color:#f87171;font-size:11px;font-weight:700;margin-bottom:4px;">BASE <span style="float:right;">%.0f/50</span></div>
            <div style="color:#8b949e;font-size:12px;line-height:1.5;">%s</div>
          </div>
          <div style="border-left:3px solid #fbbf24;padding-left:12px;">
            <div style="color:#fbbf24;font-size:11px;font-weight:700;margin-bottom:4px;">SFT <span style="float:right;">%.0f/50</span></div>
            <div style="color:#c9d1d9;font-size:12px;line-height:1.5;">%s</div>
          </div>
          <div style="border-left:3px solid #3fb950;padding-left:12px;">
            <div style="color:#3fb950;font-size:11px;font-weight:700;margin-bottom:4px;">DPO <span style="float:right;">%.0f/50</span></div>
            <div style="color:#e6edf3;font-size:12px;line-height:1.5;">%s</div>
          </div>
        </div>
      </div>
    </div>
    ''' % (i, "" if i < 2 else "", prompt,
           base_total, base_r, sft_total, sft_r, dpo_total, dpo_r)

html = '''
<div style="font-family:-apple-system,sans-serif;max-width:900px;margin:20px 0;">
  <h3 style="color:#c9d1d9;">Response Comparison (with scores)</h3>
  <p style="color:#8b949e;font-size:13px;margin-bottom:16px;">Each response scored on 5 dimensions (0-10 each, 50 max). Click to expand/collapse.</p>
  %s
</div>
''' % cards

display(HTML(html))

In [ ]:
# ============================================================
# VISUALIZATION 3: LLM Judge + Pairwise Results + Animated Bars
# ============================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

plt.style.use('dark_background')
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Chart 1: LLM Judge scores (grouped bar)
judge_dims_plot = ["Accuracy", "Clarity", "Detail", "Helpfulness", "Overall"]
x = np.arange(len(judge_dims_plot))
w = 0.25
stage_colors = [("#f87171", "Base"), ("#fbbf24", "SFT"), ("#3fb950", "DPO")]

for i, (color, label) in enumerate(stage_colors):
    key = label.lower()
    vals = [avg_llm[key][d] for d in judge_dims_plot]
    axes[0].bar(x + i * w, vals, w, color=color, label=label, edgecolor="white", linewidth=0.5)

axes[0].set_xticks(x + w)
axes[0].set_xticklabels(judge_dims_plot, fontsize=9)
axes[0].set_title("LLM-as-Judge Scores (1-10)", fontsize=14)
axes[0].set_ylabel("Score")
axes[0].set_ylim(0, 11)
axes[0].legend()
axes[0].grid(axis="y", alpha=0.2)

# Chart 2: Pairwise wins (pie chart)
win_labels = ["Base", "SFT", "DPO"]
win_vals = [wins["base"], wins["sft"], wins["dpo"]]
win_colors = ["#f87171", "#fbbf24", "#3fb950"]
wedges, texts, autotexts = axes[1].pie(
    win_vals, labels=win_labels, colors=win_colors,
    autopct=lambda p: "%d" % round(p * sum(win_vals) / 100),
    startangle=90, textprops={"color": "#c9d1d9", "fontsize": 12},
    wedgeprops={"edgecolor": "#0d1117", "linewidth": 2}
)
for at in autotexts:
    at.set_fontweight("bold")
    at.set_fontsize(14)
axes[1].set_title("Pairwise Judge: Total Wins", fontsize=14)

# Chart 3: Combined score (rule + LLM average) - horizontal bars
combined = {}
for stage in ["base", "sft", "dpo"]:
    rule_avg = np.mean(list(avg_scores[stage].values()))
    llm_avg = np.mean(list(avg_llm[stage].values()))
    combined[stage] = {"Rule-Based": rule_avg, "LLM Judge": llm_avg}

stage_labels = ["Base", "SFT", "DPO"]
stage_keys = ["base", "sft", "dpo"]
y_pos = np.arange(len(stage_labels))
bar_h = 0.35

for i, (metric, color) in enumerate([("Rule-Based", "#818cf8"), ("LLM Judge", "#d2a8ff")]):
    vals = [combined[s][metric] for s in stage_keys]
    axes[2].barh(y_pos + i * bar_h, vals, bar_h, color=color, label=metric, edgecolor="white", linewidth=0.5)
    for y, v in zip(y_pos + i * bar_h, vals):
        axes[2].text(v + 0.1, y, "%.1f" % v, va="center", fontsize=11, fontweight="bold", color=color)

axes[2].set_yticks(y_pos + bar_h / 2)
axes[2].set_yticklabels(stage_labels, fontsize=12)
axes[2].set_title("Average Scores (Rule vs LLM)", fontsize=14)
axes[2].set_xlabel("Score (0-10)")
axes[2].set_xlim(0, 11)
axes[2].legend()
axes[2].grid(axis="x", alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# VISUALIZATION 4: Animated Summary Dashboard
# ============================================================

# Compute final metrics
rule_totals = {s: sum(avg_scores[s].values()) for s in ["base", "sft", "dpo"]}
llm_totals = {s: sum(avg_llm[s].values()) for s in ["base", "sft", "dpo"]}
lengths = {s: np.mean([len(all_responses[s][p].split()) for p in TEST_PROMPTS]) for s in ["base", "sft", "dpo"]}

def animated_bar(value, max_val, color, delay):
    pct = min(100, value / max_val * 100)
    return '''<div style="background:#0d1117;border-radius:6px;overflow:hidden;height:24px;margin:4px 0;">
      <div style="width:%.0f%%;background:%s;height:100%%;border-radius:6px;
                  animation:growBar 1.2s ease-out %ss both;display:flex;align-items:center;padding-left:8px;">
        <span style="color:white;font-size:11px;font-weight:700;">%.1f</span>
      </div>
    </div>''' % (pct, color, delay, value)

stages_html = ""
for stage, label, color, delay_base in [("base", "Base", "#f87171", "0"), ("sft", "SFT", "#fbbf24", "0.3"), ("dpo", "DPO", "#3fb950", "0.6")]:
    stages_html += '''
    <div style="background:linear-gradient(135deg,#1a1a2e,#161b22);border:1px solid #30363d;
                border-radius:12px;padding:18px;animation:fadeUp 0.6s ease %ss both;">
      <div style="color:%s;font-size:15px;font-weight:700;margin-bottom:12px;">%s</div>
      <div style="font-size:11px;color:#8b949e;margin-bottom:2px;">Rule Score (out of 50)</div>
      %s
      <div style="font-size:11px;color:#8b949e;margin-bottom:2px;margin-top:8px;">LLM Judge (out of 50)</div>
      %s
      <div style="font-size:11px;color:#8b949e;margin-bottom:2px;margin-top:8px;">Pairwise Wins</div>
      %s
      <div style="font-size:11px;color:#8b949e;margin-bottom:2px;margin-top:8px;">Avg Words</div>
      %s
    </div>
    ''' % (delay_base, color, label,
           animated_bar(rule_totals[stage], 50, color, delay_base),
           animated_bar(llm_totals[stage], 50, color, str(float(delay_base)+0.1)),
           animated_bar(wins[stage], 15, color, str(float(delay_base)+0.2)),
           animated_bar(lengths[stage], 150, color, str(float(delay_base)+0.3)))

html = '''
<style>
@keyframes growBar { from { width: 0%%; } }
@keyframes fadeUp { from { opacity: 0; transform: translateY(20px); } to { opacity: 1; transform: translateY(0); } }
</style>
<div style="font-family:-apple-system,sans-serif;max-width:900px;margin:20px 0;">
  <h3 style="color:#c9d1d9;margin-bottom:16px;">Final Scorecard — All Evaluation Methods</h3>
  <div style="display:grid;grid-template-columns:repeat(3,1fr);gap:16px;">
    %s
  </div>
  <div style="background:#161b22;border:1px solid #30363d;border-radius:12px;padding:18px;margin-top:16px;">
    <table style="width:100%%;color:#c9d1d9;font-size:13px;border-spacing:0 6px;">
      <tr><td style="color:#8b949e;">Model</td><td style="text-align:right;font-weight:600;">Qwen2.5-0.5B</td></tr>
      <tr><td style="color:#8b949e;">Evaluation methods</td><td style="text-align:right;">Rule-based (5 dims) + LLM Judge (5 dims) + Pairwise Judge</td></tr>
      <tr><td style="color:#8b949e;">SFT data</td><td style="text-align:right;">2,000 Alpaca pairs</td></tr>
      <tr><td style="color:#8b949e;">DPO data</td><td style="text-align:right;">10 preference pairs</td></tr>
      <tr><td style="color:#8b949e;">Total training</td><td style="text-align:right;font-weight:600;">%.0f seconds</td></tr>
    </table>
  </div>
</div>
''' % (stages_html, sft_time + dpo_time)

display(HTML(html))

---
## The Core Ideas (What You Just Saw)

### 1. Pre-training ≠ Post-training

```
Pre-training:   "Read the entire internet"     → knows facts, grammar, patterns
Post-training:  "Learn to be an assistant"      → follows instructions, gives good answers

Pre-training costs: millions of dollars, months of compute
Post-training costs: a few dollars, hours on one GPU

Most of the "intelligence" comes from pre-training.
Post-training just teaches the model HOW to use that intelligence.
```

### 2. SFT teaches FORMAT, DPO teaches QUALITY

```
SFT:  "When someone asks X, respond with Y"     → learns the pattern
DPO:  "This answer is better than that answer"   → learns preferences

SFT makes the model helpful.
DPO makes the model better at being helpful.
```

### 3. You don't need millions of examples

```
SFT:  2,000 examples was enough to teach instruction-following
DPO:  10 preference pairs was enough to shift response quality

Why? The model already KNOWS the information from pre-training.
Post-training just teaches it how to present that knowledge.
```

### 4. The post-training pipeline in production

```
What OpenAI/Anthropic/Google actually do:

  1. Pre-train on trillions of tokens          (months, $100M+)
  2. SFT on ~100K high-quality conversations    (days, $10K)
  3. RLHF/DPO on human preference data          (days, $10K)
  4. Safety training (red-teaming, filtering)    (weeks)
  5. Repeat steps 2-4 with better data           (continuous)

What you just did in this notebook:

  1. Used a pre-trained model (Qwen2.5-0.5B)    (downloaded)
  2. SFT on 2,000 Alpaca examples               (minutes)
  3. DPO on 10 preference pairs                  (seconds)

Same pipeline. Smaller scale. Same core ideas.
```